# 08 · Síntesis: cada insight, verificado con código

Este notebook no produce análisis nuevo. Relee las tablas de `datos/processed/` y los registros manuales, recalcula cada afirmación del artículo y **falla si alguna deja de ser cierta** (por ejemplo, si se corrige un dato o se agrega una cifra nueva al registro).

Ejecútalo después de los notebooks 01-07.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.graph_objects as go

from src import config, tema
from src import procesamiento as proc

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

In [2]:
P = config.DIR_PROCESSED
poblacion = pd.read_csv(P / "poblacion_censal.csv")
cobertura = pd.read_csv(P / "cobertura_cifras_por_anio.csv")
dispersion = pd.read_csv(P / "dispersion_cifras_por_anio.csv").set_index("anio_evento")
sensibilidad = pd.read_csv(P / "multiplicador_sensibilidad.csv")
rutas = pd.read_csv(P / "rutas_osrm.csv").set_index("origen")
clima = pd.read_csv(P / "clima_ventana_fiestas.csv")
denue = pd.read_csv(P / "denue_comparativo_municipios.csv").set_index("municipio")
wiki = pd.read_csv(P / "wikipedia_indice_fiestas.csv").set_index("anio")
prensa = proc.leer_registro_prensa()
distancias = proc.leer_distancias_prensa()

insights = []

def verificar(id_, afirmacion, valor, condicion, fuente):
    insights.append({"id": id_, "afirmacion": afirmacion, "valor": valor, "verificado": bool(condicion), "fuente": fuente})
    assert condicion, f"{id_} ya no se sostiene: {afirmacion} (valor: {valor})"

In [3]:
mag = poblacion[poblacion["municipio"] == "Magdalena"].set_index("anio")
cab_2020 = int(mag.loc[2020, "pob_cabecera"])
verificar("I01", "La cabecera de Magdalena de Kino tenía 30,429 habitantes en 2020", f"{cab_2020:,}",
          cab_2020 == 30429, "INEGI ITER 2020")

crec = mag.loc[2020, "pob_cabecera"] / mag.loc[2000, "pob_cabecera"] - 1
verificar("I02", "La cabecera creció alrededor de 38% entre 2000 y 2020", f"{crec:.1%}",
          0.35 < crec < 0.40, "INEGI ITER 2000 y 2020")

base = sensibilidad[sensibilidad["denominador"] == "Cabecera, Censo 2020"]
vmin, vmax = base["veces"].min(), base["veces"].max()
verificar("I03", "Las estimaciones de visitantes equivalen a entre 2 y 5 veces la población del pueblo",
          f"{vmin:.2f}x - {vmax:.2f}x", 1.9 < vmin < 2.1 and 4.8 < vmax < 5.0, "notebook 03")

verificar("I04", "El multiplicador sigue entre ~1.7x y 5x con cualquier denominador razonable",
          f"{sensibilidad['veces'].min():.2f}x - {sensibilidad['veces'].max():.2f}x",
          sensibilidad["veces"].min() > 1.6 and sensibilidad["veces"].max() < 5.0, "notebook 03")

In [4]:
sin_dato = cobertura[(cobertura["anio"].between(2010, 2019)) & (cobertura["cifras"] == 0)]
verificar("I05", "Ningún año entre 2010 y 2019 tiene una cifra de asistencia atribuible", f"{len(sin_dato)} de 10 años",
          len(sin_dato) == 10, "registro de prensa")

balances = prensa[(prensa["que_mide"] == "visitantes_totales") & (prensa["momento"] == "balance")]
verificar("I06", "Solo existe un balance posterior de visitantes totales (60 mil, 2023)",
          f"{len(balances)} balance(s): {balances['cifra_min'].tolist()}",
          len(balances) == 1 and balances["cifra_min"].iloc[0] == 60000, "El Imparcial 10/10/2023")

veces_2025 = dispersion.loc[2025, "veces"]
verificar("I07", "En 2025 circularon cifras de 12 mil a 150 mil en la misma semana", f"{veces_2025}x",
          dispersion.loc[2025, "minimo"] == 12000 and dispersion.loc[2025, "maximo"] == 150000, "Expreso; Meganoticias")

pc = prensa[prensa["id"] == "P12"].iloc[0]
verificar("I08", "Protección Civil estatal cerró 2025 sin cifra de peregrinos atendidos",
          "sin cifra" if pd.isna(pc["cifra_min"]) else pc["cifra_min"], pd.isna(pc["cifra_min"]), "Uniradio Informa 7/10/2025")

derrama = prensa[prensa["que_mide"] == "derrama_economica"]
verificar("I09", "Hay una sola cifra de derrama económica (más de 60 mdp, 2023)", f"{len(derrama)} cifra(s)",
          len(derrama) == 1 and derrama["cifra_min"].iloc[0] == 60_000_000, "El Imparcial 10/10/2023")

In [5]:
km_nog = rutas.loc["Nogales", "km_carretera"]
nog_prensa = distancias[distancias["origen"] == "Nogales"]["km_min"]
verificar("I10", "La ruta Nogales-Magdalena mide unos 89 km por carretera, y la prensa publica 6 cifras distintas",
          f"{km_nog:.1f} km; {nog_prensa.nunique()} cifras ({nog_prensa.min():g}-{nog_prensa.max():g})",
          88 < km_nog < 91 and nog_prensa.nunique() >= 6, "OSRM; registro de distancias")

en_rango = rutas[(rutas["km_carretera"] >= 50) & (rutas["km_carretera"] <= 76)]
verificar("I11", "Ninguna ruta de origen cae en el rango de 50-76 km del brief original",
          f"{len(en_rango)} rutas en rango", len(en_rango) == 0, "OSRM")

km_persona = 25000 * km_nog
verificar("I12", "25 mil caminantes haciendo la ruta completa sumarían más de 2.2 millones de km (55 vueltas a la Tierra)",
          f"{km_persona / 1e6:.2f} M km; {km_persona / 40075:.0f} vueltas", km_persona > 2.2e6, "notebook 04 (techo)")

In [6]:
mc = clima[clima["punto"] == "magdalena"].set_index("anio")
anio_max = mc["t_max_promedio"].idxmax()
verificar("I13", "2024 fueron las fiestas más calurosas desde 1995 en Magdalena",
          f"{anio_max}: {mc.loc[anio_max, 't_max_promedio']} °C, {int(mc.loc[anio_max, 'dias_35_o_mas'])} días ≥35 °C",
          anio_max == 2024, "Open-Meteo ERA5")

h = denue.loc["Magdalena"]
verificar("I14", "Magdalena tiene 13 hoteles y moteles: más que la mediana de Sonora, pero menos de la mitad que Álamos o Puerto Peñasco",
          f"{int(h['Hospedaje'])} establecimientos; {h['Hospedaje por mil hab.']:.2f} por mil; lugar {int(h['rank_hospedaje'])} de {len(denue)}",
          int(h["Hospedaje"]) == 13
          and h["Hospedaje por mil hab."] > denue["Hospedaje por mil hab."].median()
          and h["Hospedaje por mil hab."] < denue.loc["Alamos", "Hospedaje por mil hab."] / 2,
          "INEGI DENUE")

idx = wiki["indice_fiestas"].mean()
verificar("I15", "La fiesta apenas mueve las visitas a Wikipedia (unas 1.4 veces el resto del año)",
          f"{idx:.2f}", idx < 2, "Wikimedia pageviews")

In [7]:
resultado = pd.DataFrame(insights)
config.DIR_TABLAS.mkdir(parents=True, exist_ok=True)
resultado.to_csv(config.DIR_TABLAS / "insights_verificados.csv", index=False)
print(f"{resultado['verificado'].sum()} de {len(resultado)} insights verificados")
resultado

15 de 15 insights verificados


,id,afirmacion,valor,verificado,fuente
0,I01,"La cabecera de Magdalena de Kino tenía 30,429 ...","30,429",True,INEGI ITER 2020
1,I02,La cabecera creció alrededor de 38% entre 2000...,38.2%,True,INEGI ITER 2000 y 2020
2,I03,Las estimaciones de visitantes equivalen a ent...,1.97x - 4.93x,True,notebook 03
3,I04,El multiplicador sigue entre ~1.7x y 5x con cu...,1.82x - 4.93x,True,notebook 03
4,I05,Ningún año entre 2010 y 2019 tiene una cifra d...,10 de 10 años,True,registro de prensa
5,I06,Solo existe un balance posterior de visitantes...,1 balance(s): [60000.0],True,El Imparcial 10/10/2023
6,I07,En 2025 circularon cifras de 12 mil a 150 mil ...,12.5x,True,Expreso; Meganoticias
7,I08,Protección Civil estatal cerró 2025 sin cifra ...,sin cifra,True,Uniradio Informa 7/10/2025
8,I09,Hay una sola cifra de derrama económica (más d...,1 cifra(s),True,El Imparcial 10/10/2023
9,I10,La ruta Nogales-Magdalena mide unos 89 km por ...,89.4 km; 6 cifras (80-100),True,OSRM; registro de distancias


## Lo que estos datos no pueden afirmar

- **Ninguna cifra de afluencia es un conteo.** Todas son declaraciones o proyecciones; el multiplicador es un orden de magnitud.
- **No hay tendencia.** Con un hueco total en 2010-2019 y cifras que miden cosas distintas, no se puede decir si la peregrinación crece o decrece.
- **2021 se excluye por atípico**: año de pandemia, sin fuente primaria y con versiones contradictorias sobre si hubo fiesta presencial.
- **Las distancias son por carretera** (perfil vehicular de OSRM), no el trazo peatonal.
- **El clima es reanálisis** (malla de ~25 km); compara años entre sí, no da la temperatura exacta del acotamiento.
- **El DENUE cuenta establecimientos, no cuartos.** No permite calcular capacidad hotelera.
- **Wikipedia mide curiosidad en línea**, no asistencia.